In [ ]:
import os
import pickle
import pandas as pd
import re
import nltk

# Pastikan stopwords sudah diunduh jika belum
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download("stopwords")
from nltk.corpus import stopwords

# --- Konfigurasi Path ---
# Sesuaikan base_dir jika notebook Anda tidak berada di root direktori proyek yang sama
# dengan struktur folder 'models' dan 'dataset/use'
# Jika script_dir menyebabkan error di notebook, Anda bisa set base_dir secara manual
# contoh: base_dir = "." # Jika notebook ada di root project
#         base_dir = "path/to/your/project_root"
try:
    # Coba dapatkan script_dir, mungkin tidak bekerja ideal di semua environment notebook
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Jika __file__ tidak terdefinisi (umum di notebook interaktif), set manual
    script_dir = "." # Asumsi notebook ada di root proyek, sesuaikan jika perlu
    print(f"[Info] __file__ tidak terdefinisi, menggunakan script_dir = '{script_dir}'. "
          f"Pastikan path ke model dan dataset benar.")


# --- Fungsi Preprocessing (tidak berubah) ---
stopword_set = set(stopwords.words("english")) # Ganti nama variabel agar tidak konflik

def preprocess_text(text):
    text = str(text).lower() # Pastikan input adalah string
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\b\w{1,2}\b", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = " ".join(word for word in text.split() if word not in stopword_set)
    return text

# --- Fungsi Load Model (modifikasi kecil pada path) ---
def load_content_model():
    model_content_path = os.path.join(script_dir, "models", "content.pkl")
    data_content_path = os.path.join(script_dir, "dataset", "use", "content_df.csv")

    print(f"Mencoba memuat model konten dari: {model_content_path}")
    print(f"Mencoba memuat data konten dari: {data_content_path}")

    try:
        with open(model_content_path, "rb") as file:
            model_data = pickle.load(file)
            cos_sim = model_data["cos_sim"]
            # tfidf = model_data["tfidf"] # tfidf tidak digunakan di fungsi recommendation Anda
    except FileNotFoundError:
        print(f"[Error] File model konten tidak ditemukan di {model_content_path}")
        print("Pastikan path sudah benar atau file model ada.")
        return None, None
    except Exception as e:
        print(f"[Error] Gagal memuat model konten: {e}")
        return None, None

    try:
        df = pd.read_csv(data_content_path)
    except FileNotFoundError:
        print(f"[Error] File data konten tidak ditemukan di {data_content_path}")
        print("Pastikan path sudah benar atau file dataset ada.")
        return None, None
    except Exception as e:
        print(f"[Error] Gagal memuat data konten: {e}")
        return None, None

    features = df[
        ["Title", "description", "authors", "publishedDate", "categories", "publisher"]
    ].copy() # Gunakan .copy() untuk menghindari SettingWithCopyWarning
    features["publishedYear"] = features["publishedDate"].str[:4]
    features.drop(columns=["publishedDate"], inplace=True) # Hapus kolom setelah digunakan
    features["content"] = (
        features["description"].astype(str) # Pastikan semua string
        + " "
        + features["authors"].astype(str)
        + " "
        + features["categories"].astype(str)
        + " " # Spasi sebelum publisher
        + features["publisher"].astype(str) # Tambahkan publisher seperti di dokumen
    )
    features["clean_content"] = features["content"].apply(preprocess_text)
    features.set_index(features["Title"].str.lower(), inplace=True) # Lowercase index Title
    features.index.name = "Index_Title" # Nama index yang valid

    return features, cos_sim


def load_collab_model():
    model_collab_path = os.path.join(script_dir, "models", "collab.pkl")
    data_collab_path = os.path.join(script_dir, "dataset", "use", "collaborative_df.csv")

    print(f"Mencoba memuat model kolaboratif dari: {model_collab_path}")
    print(f"Mencoba memuat data kolaboratif dari: {data_collab_path}")

    try:
        with open(model_collab_path, "rb") as file:
            model_data = pickle.load(file)
            user_sim_df = model_data["user_sim_df"]
            user_books_matrix = model_data["user_books_matrix"]
    except FileNotFoundError:
        print(f"[Error] File model kolaboratif tidak ditemukan di {model_collab_path}")
        return None, None, None
    except Exception as e:
        print(f"[Error] Gagal memuat model kolaboratif: {e}")
        return None, None, None

    try:
        df = pd.read_csv(data_collab_path)
    except FileNotFoundError:
        print(f"[Error] File data kolaboratif tidak ditemukan di {data_collab_path}")
        return None, None, None
    except Exception as e:
        print(f"[Error] Gagal memuat data kolaboratif: {e}")
        return None, None, None
        
    return df, user_books_matrix, user_sim_df

# --- Fungsi Rekomendasi (sedikit penyesuaian pada get_loc jika ada error) ---
def recomendation(name, features, cos_sim, top_n=10): # Sesuai nama fungsi di dokumen
    try:
        # Pastikan 'name' sudah di-lowercase sebelum get_loc, karena index di-lowercase saat load_content_model
        idx = features.index.get_loc(name.lower())
    except KeyError:
        print(f"[Info] Buku '{name}' tidak ditemukan dalam index fitur konten.")
        return pd.DataFrame(columns=["Recommended Books"])
        
    # score.index akan berisi integer 0..N-1 yang merupakan posisi asli buku
    # setelah diurutkan berdasarkan similarity, index ini tetap mengacu pada posisi asli
    score = pd.Series(cos_sim[idx]).sort_values(ascending=False)
    
    # top_score_indices akan berisi daftar posisi integer (0..N-1) dari buku-buku paling mirip
    top_score_indices = list(score.iloc[1 : top_n + 1].index)

    # Gunakan .iloc untuk mengakses baris berdasarkan posisi integer
    recommended_rows = features.iloc[top_score_indices] # PERBAIKAN DI SINI
    
    # Ambil 'Title' (dengan casing asli) dari baris tersebut
    recommended_books = recommended_rows["Title"].unique()
    
    return pd.DataFrame(recommended_books, columns=["Recommended Books"])


def user_recommendation(user_id, user_sim_df, user_books_matrix, df_collab, top_n=10): # Sesuai nama fungsi di dokumen [cite: 23]
    if user_id not in user_sim_df.index:
        print(f"[Info] User ID '{user_id}' tidak ditemukan dalam matriks similaritas pengguna.")
        return pd.DataFrame(columns=["Recommended Books"])
    if user_id not in user_books_matrix.index:
        print(f"[Info] User ID '{user_id}' tidak ditemukan dalam matriks interaksi pengguna-buku.")
        return pd.DataFrame(columns=["Recommended Books"])

    sim_user = user_sim_df[user_id].sort_values(ascending=False).index[1:] # Pengguna serupa
    sim_user_rating = user_books_matrix.loc[sim_user] # Rating dari pengguna serupa
    
    # Buku yang belum dirating oleh target user
    user_no_rated_indices = user_books_matrix.loc[user_id][user_books_matrix.loc[user_id] == 0].index
    
    # Pastikan kolom ada di sim_user_rating sebelum melakukan mean
    valid_cols_for_mean = [col for col in user_no_rated_indices if col in sim_user_rating.columns]
    if not valid_cols_for_mean:
        print(f"[Info] Tidak ada buku yang belum dirating oleh user {user_id} yang memiliki rating dari pengguna serupa.")
        return pd.DataFrame(columns=["Recommended Books"])

    recommendation_scores = (
        sim_user_rating[valid_cols_for_mean]
        .mean()
        .sort_values(ascending=False)
        .head(top_n)
    )
    # Ambil judul buku unik dari df_collab (dataset review asli)
    recommended_books = df_collab[df_collab["Title"].isin(recommendation_scores.index)]["Title"].unique()
    return pd.DataFrame(recommended_books, columns=["Recommended Books"])


# --- Fungsi Utama untuk Menjalankan di Jupyter Notebook ---
def run_recommendation_notebook():
    print("Memuat model dan data...")
    features_content, cos_sim_content = load_content_model()
    df_collab, user_books_matrix_collab, user_sim_df_collab = load_collab_model()

    # Memuat data asli untuk informasi buku (misal: gambar)
    # Pastikan path ini benar
    try:
        data_books_info_path = os.path.join(script_dir, "dataset", "use", "content_df.csv")
        data_books_info = pd.read_csv(data_books_info_path)
    except FileNotFoundError:
        print(f"[Warning] File info buku '{data_books_info_path}' tidak ditemukan. Detail gambar tidak akan ditampilkan.")
        data_books_info = None
    
    # Cek apakah model berhasil dimuat
    if features_content is None or df_collab is None:
        print("Gagal memuat satu atau lebih model/data. Proses rekomendasi dihentikan.")
        return

    # --- Input Pengguna (Contoh, bisa diganti dengan input()) ---
    # Ambil contoh User ID dan Judul Buku dari data yang ada
    try:
        sample_user_id = df_collab["User_id"].unique()[0]
        # Ambil judul buku dari 'features_content' yang index-nya sudah lowercase
        # tapi kita butuh judul dengan casing asli untuk input ke fungsi 'recomendation'
        # dan untuk dicari di 'data_books_info'
        sample_book_title_lowercase = features_content.index[0] # Ini sudah lowercase
        # Cari judul asli dari kolom 'Title' di features_content berdasarkan index lowercase
        sample_book_title_original_case = features_content.loc[sample_book_title_lowercase, "Title"]

    except IndexError:
        print("[Error] Tidak dapat mengambil contoh User ID atau Judul Buku dari data. Pastikan dataset tidak kosong.")
        return

    selected_user_id = input(f"Masukkan User ID (contoh: {sample_user_id}): ") or sample_user_id
    selected_book_title = input(f"Masukkan Judul Buku yang Diinginkan (contoh: {sample_book_title_original_case}): ") or sample_book_title_original_case
    
    print(f"\nMemproses rekomendasi untuk User ID: {selected_user_id} dan Buku: {selected_book_title}...")

    # Dapatkan Rekomendasi
    collab_recommendation = user_recommendation(
        selected_user_id, user_sim_df_collab, user_books_matrix_collab, df_collab, top_n=10
    )
    # Fungsi 'recomendation' mengharapkan 'name' (judul buku) dalam casing asli,
    # karena di dalamnya akan di .lower() untuk dicocokkan dengan index.
    item_recommendation = recomendation(selected_book_title, features_content, cos_sim_content, top_n=15)

    # Gabungkan Rekomendasi (Hybrid)
    hybrid_recommendation = (
        pd.concat([collab_recommendation, item_recommendation])
        .drop_duplicates(subset="Recommended Books")
        .sample(frac=1, random_state=42) # Sesuai dokumen [cite: 24]
        .head(15) # Sesuai dokumen [cite: 24]
    )

    print("\n--- Rekomendasi Hybrid (Gabungan dari Aktivitas Baca Anda & Buku Terkait) ---")
    if not hybrid_recommendation.empty:
        for i, title in enumerate(hybrid_recommendation["Recommended Books"]):
            book_image_info = "Tidak ada info gambar"
            if data_books_info is not None and title in data_books_info["Title"].values:
                book_image = data_books_info[data_books_info["Title"] == title]["image"].values
                if len(book_image) > 0:
                    book_image_info = f"(Gambar: {book_image[0]})"
            print(f"{i+1}. {title} {book_image_info}")
    else:
        print("Tidak ada rekomendasi hybrid yang bisa ditampilkan.")

    print("\n--- Buku Terkait Lainnya (Dari kemiripan konten) ---")
    # Buku dari item_recommendation yang tidak ada di hybrid_recommendation
    other_related_books = item_recommendation[
        ~item_recommendation["Recommended Books"].isin(
            hybrid_recommendation["Recommended Books"]
        )
    ]
    if not other_related_books.empty:
        for i, title in enumerate(other_related_books["Recommended Books"]):
            book_image_info = "Tidak ada info gambar"
            if data_books_info is not None and title in data_books_info["Title"].values:
                book_image = data_books_info[data_books_info["Title"] == title]["image"].values
                if len(book_image) > 0:
                    book_image_info = f"(Gambar: {book_image[0]})"
            print(f"{i+1}. {title} {book_image_info}")
    else:
        print("Tidak ada buku terkait lainnya untuk ditampilkan.")

if __name__ == "__main__":
    # Jalankan fungsi untuk notebook
    run_recommendation_notebook()

[Info] __file__ tidak terdefinisi, menggunakan script_dir = '.'. Pastikan path ke model dan dataset benar.
Memuat model dan data...
Mencoba memuat model konten dari: .\models\content.pkl
Mencoba memuat data konten dari: .\dataset\use\content_df.csv
Mencoba memuat model kolaboratif dari: .\models\collab.pkl
Mencoba memuat data kolaboratif dari: .\dataset\use\collaborative_df.csv

Memproses rekomendasi untuk User ID: A140XH16IKR4B0 dan Buku: The Rabbi's Cat...


KeyError: "None of [Index([311, 950, 728, 518, 825, 639, 122, 273, 815, 957, 586, 239, 793, 636,\n       763],\n      dtype='object', name='Index_Title')] are in the [index]"